# RadioSkySims — Simulação de TOD (leve)

Pipeline de geração de Time-Ordered Data (TOD) do RadioSkySim **desacoplado**
(`radioskysims_simulation`), usando a API `SimOrchestrator` e o perfil
`uirapuru` do `settings.toml`.

Este notebook é uma **simulação leve**: janela de 1 h (passo 60 s), apenas GPS,
com `use_processor=True` para batches seguros de memória. Valida o ciclo
**satélites → TOD → persistência** sem travar a máquina (a versão com 24 h e
4 constelações sobrecarregou o ambiente anteriormente).

Fluxo:
1. Parâmetros de referência (leves)
2. Radiotelescópio (`uirapuru`) e orquestrador
3. Satélites GPS com PRN (refatoração NORAD ↔ Nome ↔ PRN)
4. Simulação TOD discreta (GPS)
5. Persistência HDF5/NetCDF + reload (ciclo "ida e volta")


## 1. Parâmetros de referência (leves)

Janela curta (1 h, passo 60 s) e observador do radiotelescópio `uirapuru`.
A data de referência é 1 mês antes de agora (UTC).


In [ ]:
# ── Parâmetros de referência para a simulação leve ──
from datetime import UTC, datetime, timedelta
from pathlib import Path

import numpy as np  # noqa: F401

from radioskysims import create_radiotelescope
from radioskysims_core.domain.common import FrequencyRange, TimeRange

# Data de referência: 1 mês antes de agora (UTC).
REF_DATE = (datetime.now(UTC) - timedelta(days=30)).replace(microsecond=0)
print("Data de referência (REF_DATE):", REF_DATE.isoformat())

# Janela LEVE: 1 h a partir de REF_DATE, passo de 60 s.
_ISO = "%Y-%m-%dT%H:%M:%S"
TR = TimeRange.from_iso(
    REF_DATE.strftime(_ISO),
    (REF_DATE + timedelta(minutes=60)).strftime(_ISO),
    60,
)
print("TimeRange (1 h, 60 s):", TR.to_datetime64().shape[0], "instantes")

FR = FrequencyRange(min_mhz=1000.0, max_mhz=1250.0, resolution_mhz=0.5)

# Observador do radiotelescópio (perfil uirapuru em settings.toml).
uirapuru = create_radiotelescope("uirapuru")
OBSERVER = uirapuru.observer
print("Observador:", OBSERVER)

# Diretório de saída.
TOD_DIR = Path("data") / "TOD"
TOD_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ── Orquestrador de simulação (API desacoplada) ──
from radioskysims_simulation.application.services.sim_orchestrator import (
    SimOrchestrator,
)

orch = SimOrchestrator.from_settings(telescope="uirapuru")
print("✅ Orquestrador configurado")
print("Constelações padrão (settings):", orch.default_constellations())
print("Time range padrão (settings):", orch.default_time_range())


## 2. Satélites GPS com PRN

A refatoração integrou o mapeamento **NORAD ↔ Nome ↔ PRN** ao modelo persistido
(ver `radioskysims-sources`, branch `feature/satellite-prn-mapping`). Cada
satélite agora expõe `prn` e `sv` (ex.: `G22`), facilitando a comparação com
RINEX e com o catálogo de sinais.


In [ ]:
# ── Satélites GPS via SatelliteManager (TLE do banco) ──
from radioskysims_core.db.engine import Database
from radioskysims_sources.application.services.satellite_manager import (
    SatelliteManager,
)
from radioskysims_sources.bootstrap.builders import build_repositories

db = Database()
repos = build_repositories(db)
mgr = SatelliteManager(
    repository=repos["satellite_repository"],
    signal_repository=repos["signal_repository"],
    epoch=REF_DATE,
)

sats = mgr.create_from_constellations(["GPS"])
print(f"✅ {len(sats)} satélites GPS carregados (TLE na época de referência)")

# NORAD ↔ Nome ↔ PRN (refatoração integrada ao modelo persistido).
print(f"{'NORAD':>8} | {'Nome':<28} | {'PRN':>4} | SV")
for sat in sats[:10]:
    prn = sat.prn if sat.prn is not None else None
    sv = sat.sv if sat.prn is not None else None
    print(f"{sat.norad_id:>8} | {sat.name:<28} | {prn if prn is not None else '—':>4} | {sv if sv is not None else '—'}")


## 3. Simulação TOD (discreta, leve)

Gera o TOD dos satélites GPS sobre o pointing do `uirapuru`, usando o
`ResourceAwareProcessor` (`use_processor=True`) para batches seguros de memória.
O dataset permanece **lazy (dask)** — não materializamos o TOD completo em RAM.


In [ ]:
# ── Simulação TOD discreta (GPS, janela de 1 h) ──
import time

t0 = time.monotonic()
result = orch.run(
    constellations=["GPS"],
    time_range=TR,
    include_satellites=True,
    include_local_sources=False,
    include_radio_sources=False,
    use_processor=True,
)
print(f"⏱ run() completou em {time.monotonic() - t0:.1f}s")

ds = result.simulation_result.dataset
print("dims:", dict(ds.sizes))
print("variáveis:", list(ds.data_vars))

# Resumo (lazy-safe — não computa o dataset completo).
print(result.format_summary()[:1200])


In [ ]:
# ── Persistência + reload (ciclo "ida e volta") ──
from radioskysims_simulation.domain.simulation import (
    SimulationOrchestratorResult,
)

out = str(TOD_DIR / "gps_tod_leve.nc")
sidecar = result.save_result(out)
print("Salvo dataset:", out)
print("Sidecar JSON :", sidecar)

# Reload sem re-rodar a simulação:
loaded = SimulationOrchestratorResult.load_result(out)
print("Reload OK. Resumo:")
print(loaded.format_summary()[:600])


## 4. Próximos passos (ciclo ida e volta completo)

A simulação leve valida o ciclo **satélites → TOD → persistência**. Para o
ciclo completo "ida e volta" (mapas → TOD e TOD → mapas), adicione células
opcionais abaixo — ligue as flags apenas quando houver recurso:

- **Galáxia (PySM3)**: `orch.run(include_galaxy=True, galaxy_nside=64, persist_skymaps=True)` → projeta mapas HEALPix no pointing (parte pesada; por padrão desligada).
- **NVSS**: `include_radio_sources=True` com filtro pela declinação do pointing.
- **Comparação TLE vs RINEX**: mesmo satélite nos dois tracks (`gnsstools4astro`).
- **Waterfall**: visualização do TOD com `radioskysims_waterfall`.
